# Get geographic location from affiliation without ROR

In [ ]:
!python -m spacy download fr_core_news_md
!python -m spacy download en_core_web_md

In [ ]:
#chargement des modules
import pandas as pd # pour le dataframe

import json # pour lire les données de l'api au format json

import numpy as np #outils mathématiques
import re #regex

import spacy

In [ ]:
import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [ ]:
import requests
from requests.exceptions import HTTPError
import urllib.parse
import tqdm

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_author_affil_by_publi.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep=",")


In [ ]:
df0["row_id"] = df0.id +"_"+df0.index.astype(str)
df01 = df0[["id","full_name","row_id","rors"]].copy() # tableau pour prendre uniquement les rors
df01["ror"] = df01.rors.str.split("|")
df_rors = df01.explode("ror").reset_index().drop(columns=["index"])
df_rors0 = df_rors.loc[df_rors.ror =="no ror"]
df_rors1 = df_rors.loc[df_rors.ror !="no ror"]
list_affil_without_ror = [x for x in df_rors0.row_id.unique()]
len(list_affil_without_ror)



In [ ]:
df02 = df0[["id","full_name","row_id","rors", "address"]].loc[df0.rors == "no ror"]
df02["adresse"] = df02.address.str.split("|")
df_ad = df02.explode("adresse").reset_index().drop(columns=["index"])
df_ad = df_ad.loc[df_ad.adresse != "no address"]
df_ad

In [ ]:


ROR_API_ENDPOINT = "https://api.dev.ror.org/organizations"


In [ ]:
list_ad = [x for x in df_ad.adresse.unique()]
len(list_ad)

In [ ]:
def request_ror_api(query):
    response = requests.get(ROR_API_ENDPOINT + '?' + f'query="{query.lower()}"').json()
    try:
        if response["number_of_results"] > 1 :
            item = response["items"][0]
            #print(item)
            if 'education' in item["types"] and 'funder' in item["types"] :
                rors = item["id"]
            else:
                for n, x in enumerate(response["items"]):
                    if 'education' in x["types"] and 'funder' in x["types"] :
                        datas = response["items"][n]
                        rors = datas["id"]
                    else:
                        rors = "not found"
        elif response["number_of_results"] == 1 :
            datas = response["items"][0]
            rors = datas["id"]
        else:
            rors = "not found"
    except:
        rors = "not found"

    return rors
    

In [ ]:
nlp = spacy.load("en_core_web_md")

In [ ]:
dict_ror = {}

for adresse in tqdm.tqdm(list_ad, total = len(list_ad)):
    #print("######", row.address)
    clean_address= re.sub(r"\s-\s", ". ", adresse)
    doc = nlp(clean_address)
    list_ror = []
    for ent in doc.ents:
        #print(ent.text)
 
        search_univ = re.findall(r"(?:universit*|coll[èe]ge|institute*|école|school)", ent.text.lower())


            
        if len(search_univ) > 0:
            #print(ent.text, ent.label_)
            rors = request_ror_api(ent.text)
            if rors != "not found":
                list_ror.append(rors)
    if len(list_ror) > 0:
        dict_ror[adresse] = "|".join(list(set([ror for ror in list_ror])))
            

dict_ror

In [ ]:
list_ad2 = [x for x in list_ad if x not in dict_ror.keys()]
len(list_ad2)


In [ ]:
nlp = spacy.load("fr_core_news_md")

In [ ]:
sent = 'Université de Paris 8 Vincennes Saint-Denis'
doc = nlp(sent)
for ent in doc.ents:
    print(ent.text, ent.label_)

In [ ]:

for adresse in tqdm.tqdm(list_ad2, total = len(list_ad2)):
    #print("######", row.address)
    clean_address= re.sub(r"\s-\s", ". ", adresse)
    doc = nlp(clean_address)
    list_ror = []
    for ent in doc.ents:
        #print(ent.text)
 
        search_univ = re.findall(r"(?:universit*|coll[èe]ge|institut*|[ée]cole|school)", ent.text.lower())


            
        if len(search_univ) > 0:
            #print(ent.text, ent.label_)
            rors = request_ror_api(ent.text)
            if rors != "not found":
                list_ror.append(rors)
    if len(list_ror) > 0:
        dict_ror[adresse] = "|".join(list(set([ror for ror in list_ror])))
            

dict_ror

In [ ]:
len(dict_ror)

In [ ]:
df_ad["ror"] = df_ad.adresse.map(dict_ror.get)
df_ad.loc[~df_ad.ror.isna()]

In [ ]:

list_ad3 = [x for x in list_ad if x not in dict_ror.keys()]
list_ad3

In [ ]:
dict_ror2 = {}
for adresse in tqdm.tqdm(list_ad3, total = len(list_ad3)):
    #print("######", row.address)
    clean_address= re.sub(r"\s-\s", ". ", adresse)
    doc = nlp(clean_address)
    list_ror = []
    for ent in doc.ents:
 
        search_univ = re.findall(r"(?:universit*|coll[èe]ge|institut*|[ée]cole|school)", ent.text.lower())


            
        if len(search_univ) > 0:
            #print(ent.text, ent.label_)
            rors = request_ror_api(ent.text)
            if rors != "not found":
                list_ror.append(rors)
    if len(list_ror) > 0:
        dict_ror[adresse] = "|".join(list(set([ror for ror in list_ror])))
        dict_ror2[adresse] = "|".join(list(set([ror for ror in list_ror])))
            

dict_ror2

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_ror_with_geographic_info.csv", "r") as file_in:
    df_inst = pd.read_csv(file_in, sep=",")

In [ ]:
df_rors = pd.concat([df_rors1, df_ad.loc[~df_ad.ror.isna()].drop(columns=["address","adresse"])])

df_rors

In [ ]:
ROR_API_ENDPOINT = "https://api.dev.ror.org/organizations" # url de l'api ROR


In [ ]:
dict_ror3 = {}
list_row = []
for ror in tqdm.tqdm(df_rors.ror.unique(), total = df_rors.ror.nunique()):
    dict_row ={}
    rorid = ror.split("/")[-1]
    #print(rorid)
    dict_ror3[ror] = rorid
    dict_row["ror"] = ror
    data = requests.get(ROR_API_ENDPOINT + '/' + rorid).json()

    for n, x in enumerate(data["names"]):
        if 'ror_display' in x["types"]:
            dict_row["nom"] = x["value"]
    locations_data = data["locations"][0]["geonames_details"]
    for i in locations_data:
        dict_row[i] = locations_data[i]
        if i == "name":
            dict_row["city"] = locations_data[i]
    dict_row["geonames_id"] = data["locations"][0]["geonames_id"]
    list_row.append(dict_row)

In [ ]:
df_inst.loc[(df_inst["name"]== "Saint-Denis") & (df_inst["country_name"]=="France")]

In [ ]:
df_inst = pd.DataFrame.from_dict(list_row)
df_inst.loc[df_inst["nom"]== "Université Paris 8", "name"]= "Saint-Denis"
df_inst.loc[df_inst["nom"]== "Université Paris 8", "city"]= "Saint-Denis"
df_inst.loc[df_inst["nom"]== "Université Paris 8", "lat"]= 48.93564
df_inst.loc[df_inst["nom"]== "Université Paris 8", "lng"]= 2.35387


In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-11_ror_with_geographic_info.csv", "w") as file_out:
    df_inst.to_csv(file_out, sep=",", index= False)

In [ ]:
df_rors01 = df_rors.merge(df_inst[['ror', 'nom', 
       'country_name', 'city', 'geonames_id']], on= ["ror"], how = "left")
df_rors01

In [ ]:
with fs.open(f"{BUCKET_OUT}/Data_bso/outputs/enriched_data/geolocation/2026-03-20_affiliation_with_info.csv", "w") as file_out:
    df_rors01.to_csv(file_out, sep=",", index= False)

In [ ]:
response = requests.get(ROR_API_ENDPOINT + '?' + f'query="university of lisbon"').json()
request_ror_api("Université de Francfort")